# Notebook 2: Record the live data once (needs internet, about 1 hour)

This notebook **downloads live rain, river and flood-report data once and saves it as files** (`fixtures`). The app then works offline from those files and always shows *'Offline demo: live data recorded on <date>'*.

Before you start:
1. Download the Natural Earth **Admin 1 – States, Provinces (10m)** zip from naturalearthdata.com, unzip it, and put the folder path of the `.shp` file in Cell 1.
2. You need internet while running Cells 4-7. Cells 1-3 work offline.
3. The free Open-Meteo service allows a limited number of calls per day. This notebook makes roughly 1,500-2,500 calls. **Do not run the recording cells more than once a day.** Finished files are skipped if you re-run.

In [ ]:
# CELL 1: settings (edit the shapefile path if yours is different)
import os, json, time
from datetime import datetime, timedelta, timezone
from pathlib import Path
import numpy as np, pandas as pd
import requests, geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

NE_SHP = Path(os.environ.get("FLOOD_NE_SHP", str(Path.cwd() / "Data" / "natural_earth" / "ne_10m_admin_1_states_provinces.shp")))
WORK = Path(os.environ.get("FLOOD_WORK", str(Path.cwd() / "work")))
FIX = WORK / "fixtures"; FIX.mkdir(parents=True, exist_ok=True)

PILOT_STATES = ["Assam", "Kerala", "Bihar", "Maharashtra", "West Bengal"]   # states with a full report
PAUSE = float(os.environ.get("FLOOD_PAUSE", "1.0"))                          # seconds between calls (be polite to the free service)
NOW = datetime.now(timezone.utc)

def get_json(url, params=None, tries=4):
    last = None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 429:
                time.sleep(20 * (k + 1)); continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e; time.sleep(3 * (k + 1))
    raise RuntimeError(f"request failed: {url} ({last})")

def save(name, obj):
    obj["recorded_at"] = NOW.isoformat(timespec="seconds")
    (FIX / name).write_text(json.dumps(obj))
    print("saved", name, f"({(FIX / name).stat().st_size / 1e3:.0f} KB)")
print("shapefile:", NE_SHP, "| exists:", NE_SHP.exists())
print("fixtures go to:", FIX)

In [ ]:
# CELL 2: load India's states from Natural Earth
G = gpd.read_file(NE_SHP)
G.columns = [c.lower() for c in G.columns]
IND = G[G["admin"] == "India"].copy().reset_index(drop=True)
print(len(IND), "India states/UTs found")
print(sorted(IND["name"]))
missing = [s for s in PILOT_STATES if s not in set(IND["name"])]
print("pilot states missing from the shapefile (fix the spelling in Cell 1):", missing)

In [ ]:
# CELL 3: choose probe points (a regular lattice of points inside each state)
def lattice_points(geom, step=0.5, cap=40):
    minx, miny, maxx, maxy = geom.bounds
    xs = np.arange(np.floor(minx / step) * step, maxx + step, step)
    ys = np.arange(np.floor(miny / step) * step, maxy + step, step)
    pg = prep(geom)
    pts = [(round(float(y), 4), round(float(x), 4)) for x in xs for y in ys if pg.contains(Point(x, y))]
    if not pts:
        p = geom.representative_point(); pts = [(round(p.y, 4), round(p.x, 4))]
    if len(pts) > cap:
        idx = np.linspace(0, len(pts) - 1, cap).astype(int); pts = [pts[i] for i in idx]
    return pts

PROBES = {r["name"]: lattice_points(r.geometry, 0.5, 40) for _, r in IND.iterrows() if r["name"] in PILOT_STATES}
NATIONAL = {r["name"]: lattice_points(r.geometry, 1.0, 10) for _, r in IND.iterrows()}
print({k: len(v) for k, v in PROBES.items()})
print("national lattice points:", sum(len(v) for v in NATIONAL.values()))

In [ ]:
# CELL 4: record rain (last 61 days + next 3 days) for the pilot states, and a coarse national grid (last 3 + next 3 days)
def fetch_rain(points, past_days, forecast_days, chunk=20):
    out = []
    for k in range(0, len(points), chunk):
        part = points[k:k + chunk]
        data = get_json("https://api.open-meteo.com/v1/forecast", dict(
            latitude=",".join(str(p[0]) for p in part), longitude=",".join(str(p[1]) for p in part),
            daily="precipitation_sum", past_days=past_days, forecast_days=forecast_days, timezone="Asia/Kolkata"))
        data = data if isinstance(data, list) else [data]
        for (lat, lon), d in zip(part, data):
            out.append(dict(lat=lat, lon=lon, dates=d["daily"]["time"], precip_mm=d["daily"]["precipitation_sum"]))
        time.sleep(PAUSE)
    return out

for state, pts in PROBES.items():
    fn = f"rain_{state.replace(' ', '_')}.json"
    if (FIX / fn).exists():
        print("skip", fn); continue
    save(fn, dict(state=state, past_days=61, forecast_days=3, probes=fetch_rain(pts, 61, 3)))

if not (FIX / "rain_national.json").exists():
    flat = [(s, p) for s, pts in NATIONAL.items() for p in pts]
    res = fetch_rain([p for _, p in flat], 3, 3)
    for (s, _), r in zip(flat, res): r["state"] = s
    save("rain_national.json", dict(past_days=3, forecast_days=3, probes=res))

In [ ]:
# CELL 5: river points for Assam. The coordinates below are APPROXIMATE: check each on Google Maps (right-click the river
# to copy its latitude/longitude) and edit them. The code then tries 9 nearby spots and keeps the one with the most flow.
ASSAM_RIVER_POINTS = [
    ("Brahmaputra near Dibrugarh", 27.48, 94.91),
    ("Brahmaputra near Neamatighat", 26.87, 94.18),
    ("Brahmaputra near Tezpur", 26.63, 92.80),
    ("Brahmaputra near Guwahati", 26.18, 91.75),
    ("Brahmaputra near Goalpara", 26.17, 90.62),
    ("Brahmaputra near Dhubri", 26.02, 89.98),
    ("Barak near Silchar", 24.83, 92.78),
]

def flood_call(lat, lon, past, fut, variables="river_discharge"):
    return get_json("https://flood-api.open-meteo.com/v1/flood", dict(
        latitude=lat, longitude=lon, daily=variables, past_days=past, forecast_days=fut))

def snap(lat, lon):
    best, best_mean = (lat, lon), -1.0
    for dy in (-0.05, 0.0, 0.05):
        for dx in (-0.05, 0.0, 0.05):
            la, lo = round(lat + dy, 4), round(lon + dx, 4)
            try:
                q = [v for v in flood_call(la, lo, 30, 1)["daily"]["river_discharge"] if v is not None]
            except Exception:
                q = []
            m = float(np.mean(q)) if q else -1.0
            if m > best_mean: best, best_mean = (la, lo), m
            time.sleep(PAUSE)
    return best, best_mean

if not (FIX / "rivers_Assam.json").exists():
    rivers = []
    for name, lat, lon in ASSAM_RIVER_POINTS:
        (la, lo), mean_q = snap(lat, lon)
        if mean_q <= 0:
            print("no river found near", name, "- skipped"); continue
        try:
            d = flood_call(la, lo, 61, 7, "river_discharge,river_discharge_p25,river_discharge_p75")
        except Exception:
            d = flood_call(la, lo, 61, 7, "river_discharge")          # fall back if the spread variables are not available
        rivers.append(dict(name=name, lat=la, lon=lo, daily=d["daily"], mean_30d=mean_q))
        print(f"{name}: snapped to {la}, {lo} (30-day mean {mean_q:.0f} m3/s)")
        time.sleep(PAUSE)
    save("rivers_Assam.json", dict(state="Assam", past_days=61, forecast_days=7, rivers=rivers))
else:
    print("skip rivers_Assam.json")

In [ ]:
# CELL 6: record flood reports for India from GDACS (last 90 days, in 30-day windows)
def gdacs_floods(start, end):
    url = ("https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH?eventlist=FL"
           f"&fromdate={start:%Y-%m-%d}&todate={end:%Y-%m-%d}&alertlevel=green;orange;red")
    return get_json(url).get("features", [])

def is_india(p):
    if "india" in str(p.get("country") or "").lower() or p.get("iso3") == "IND":
        return True
    return any((c.get("iso3") == "IND") for c in (p.get("affectedcountries") or []) if isinstance(c, dict))

if not (FIX / "gdacs_floods_india.json").exists():
    events, seen = [], set()
    for k in range(3):
        end = NOW - timedelta(days=30 * k); start = end - timedelta(days=30)
        for f in gdacs_floods(start, end):
            p = f.get("properties", {})
            key = (p.get("eventid"), p.get("episodeid"))
            if key in seen or not is_india(p): continue
            seen.add(key)
            coords = (f.get("geometry") or {}).get("coordinates") or [None, None]
            events.append(dict(eventid=p.get("eventid"), name=p.get("name"), country=p.get("country"),
                               alertlevel=p.get("alertlevel"), fromdate=p.get("fromdate"), todate=p.get("todate"),
                               lon=coords[0], lat=coords[1], link=(p.get("url") or {}).get("report")))
        time.sleep(PAUSE)
    save("gdacs_floods_india.json", dict(window_days=90, events=events))
    print(len(events), "India flood events found in the last 90 days (0 is possible and fine)")
else:
    print("skip gdacs_floods_india.json")

In [ ]:
# CELL 7: check what was recorded
for f in sorted(FIX.glob("*.json")):
    d = json.loads(f.read_text())
    n = len(d.get("probes", d.get("rivers", d.get("events", []))))
    print(f"{f.name:32s} recorded {d['recorded_at']}  items: {n}")
d = json.loads((FIX / "rain_Assam.json").read_text())
p = d["probes"][0]
print("\nAssam probe", p["lat"], p["lon"], "-> last 7 days + next 3:")
print(pd.DataFrame({"date": p["dates"], "mm": p["precip_mm"]}).tail(10).to_string(index=False))